# B3 false-positive breakdown (paper section C2)

**Descriptive only.** It does not change the GT, the metrics, the matching or any threshold, so precision stays 0.666. It reads the saved B3 results of the ablation run (commit `fcec84f`) and needs **no models** and no GPU.

| Step | What it does |
|---|---|
| 1 | Selects every B3 FP with `best_iou_any <= 0.1` (rule fixed in advance), draws contact sheets, writes `fp_tags.csv` with blank tags |
| 2 | (by hand) one tag per FP: `ROAD`, `PATCH`, `OBJ`, `UA`, `UNC` |
| 3 | Reads the tagged `fp_tags.csv` and computes the count table, the UA upper-bound precision, and example IDs |

Run the file exactly as committed. Step 1 never overwrites an existing `fp_tags.csv`, so tags cannot be lost.

In [ ]:
# ============================================================
# Cell 1 — Paths, settings, provenance
# ============================================================
import sys, os, re, csv, json, hashlib, subprocess
from datetime import datetime
from pathlib import Path

NOTEBOOK_VERSION = "b3_fp_analysis v1.0 (2026-10-11)"

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
paths_file = REPO_ROOT / "configs" / "paths.local.json"
raw = json.loads(paths_file.read_text(encoding="utf-8-sig"))
PATHS = {k: (Path(v) if Path(v).is_absolute() else (REPO_ROOT / v).resolve())
         for k, v in raw.items() if not k.startswith("_")}
EVAL_IMG_DIR  = Path(PATHS["eval_images_dir"])
GT_LABELS_DIR = Path(PATHS["gt_labels_dir"])

ABL_DIR  = REPO_ROOT / "results" / "baseline_ablation"
B3_DIR   = ABL_DIR / "B3"
FP_DIR   = ABL_DIR / "fp_analysis"
SHEET_DIR = FP_DIR / "sheets"
TAGS_CSV = FP_DIR / "fp_tags.csv"

# ---------------- SETTINGS (fixed in advance) ----------------
FP_IOU_MAX      = 0.1     # select FP rows with best_iou_any <= 0.1
BOUNDARY_RANGE  = "0.1 < best_iou_any < 0.5"
CROP_PAD_FRAC   = 0.30    # crop = predicted-polygon bounding box enlarged by 30%, made square
CROP_MIN_PX     = 128     # ... but never smaller than 128 px
TILE_PX         = 256     # crop resized to 256 x 256
THUMB_PX        = 192     # whole-image thumbnail beside each crop (GT outlines in green, FP box in red)
TILE_GAP_PX     = 14      # white gap between tiles
GRID_COLS, GRID_ROWS = 4, 5
MAX_SHEETS      = None    # None = all sheets
ALLOWED_TAGS    = ["ROAD", "PATCH", "OBJ", "UA", "UNC"]

def _git(*args):
    return subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()

try:
    GIT_COMMIT = _git("rev-parse", "--short", "HEAD")
    if _git("status", "--porcelain", "--", "notebooks/b3_fp_analysis.ipynb"):
        GIT_COMMIT += "+notebook-uncommitted"
except Exception:
    GIT_COMMIT = "unknown"

B3_METRICS = json.loads((B3_DIR / "metrics.json").read_text())
SOURCE_COMMIT = B3_METRICS["provenance"]["git_commit"]
assert SOURCE_COMMIT == "fcec84f", f"Unexpected source run commit {SOURCE_COMMIT}"
FP_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook commit :", GIT_COMMIT, "|", NOTEBOOK_VERSION)
print("Source B3 run   :", SOURCE_COMMIT, "| TP/FP/FN =", B3_METRICS["TP"], B3_METRICS["FP"], B3_METRICS["FN"])
print("Output folder   :", FP_DIR)

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 3. DO NOT EDIT. <<<
# ============================================================
# Cell 3 — Generic utilities: IoU, overlays, masks, GT reader
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# ------------ BOX IOU / NMS ------------
def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    if union <= 0:
        return 0.0
    return inter / union

def nms_boxes(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return boxes, scores
    boxes = boxes.copy()
    scores = scores.copy()
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([iou_xyxy(boxes[i], boxes[j]) for j in rest])
        rest = rest[ious < iou_thr]
        order = rest
    keep = np.array(keep, dtype=int)
    return boxes[keep], scores[keep]

# ------------ LABEL PATH → IMAGE PATH (Roboflow-style) ------------
def label_to_image_path(label_path: Path, img_dir: Path):
    """
    Map Roboflow-style labels to image filenames:
      img_0002_jpg.rf.xxxxx.txt  -> <img_dir>/img_0002.jpg (or .png/.jpeg)
    """
    stem = label_path.stem
    img_stem = stem
    for marker in ["_jpg.rf.", "_png.rf.", "_jpeg.rf."]:
        if marker in stem:
            img_stem = stem.split(marker)[0]
            break

    candidates = [img_dir / f"{img_stem}{ext}" for ext in IMG_EXTS]
    for p in candidates:
        if p.exists():
            return p

    print(f"[WARN] No image found matching label {label_path.name}")
    return None

# ------------ OVERLAYS ------------
def save_overlay_boxes_and_masks(
    img_path: Path,
    boxes_xyxy,
    masks,
    out_dir: Path,
):
    """
    Draw predicted masks (transparent red) + boxes (green).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(img_path))
    if img is None:
        return
    overlay = img.copy()

    # masks
    if masks is not None:
        for m in masks:
            m_bin = (m > 0).astype(np.uint8)
            if m_bin.sum() == 0:
                continue
            color = np.zeros_like(img)
            color[:, :, 2] = 255  # red
            mask_3 = m_bin[..., None]
            overlay = np.where(
                mask_3 == 1,
                (0.2 * color + 0.8 * overlay).astype(overlay.dtype),
                overlay,
            )

    # boxes
    if boxes_xyxy is not None and len(boxes_xyxy) > 0:
        for (x1, y1, x2, y2) in boxes_xyxy.astype(int):
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

    out_path = out_dir / img_path.name
    cv2.imwrite(str(out_path), overlay)

# ------------ MASK METRICS ------------
def mask_iou(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    if union == 0:
        return 0.0
    return inter / union

def mask_dice(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    s1 = m1.sum()
    s2 = m2.sum()
    denom = s1 + s2
    if denom == 0:
        return 0.0
    return 2.0 * inter / denom

# ------------ MASK NMS ------------
# Both functions return the INDICES of the masks to keep, so the caller can keep
# masks, boxes and scores aligned (fix: previously only masks were filtered).
def mask_nms_indices(masks, scores, iou_thr=0.7):
    """Greedy mask-level NMS. Returns indices of kept masks, highest score first."""
    if len(masks) == 0:
        return []
    scores = np.asarray(scores, dtype=float)
    order = list(scores.argsort()[::-1])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(int(i))
        order = [j for j in order if mask_iou(masks[i], masks[j]) <= iou_thr]
    return keep


def nested_mask_indices(masks, containment_thr=0.9, area_ratio_thr=0.7):
    """
    Drop 'child' masks that lie mostly inside a larger 'mother' mask.

    A child is dropped when:
      - containment = |child AND mother| / |child| >= containment_thr, and
      - area(child) / area(mother) <= area_ratio_thr.

    Fix: the previous version tested IoU >= 0.7 together with area ratio <= 0.7.
    Because IoU <= area ratio for any pair, both could only hold at exactly 0.7,
    so the rule almost never fired. Containment measures "inside" correctly.
    """
    n = len(masks)
    if n <= 1:
        return list(range(n))
    bins  = [(m > 0) for m in masks]
    areas = np.array([b.sum() for b in bins], dtype=float)
    keep  = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j or not keep[i] or not keep[j]:
                continue
            if areas[i] > areas[j] or (areas[i] == areas[j] and i < j):
                larger, smaller = i, j
            else:
                larger, smaller = j, i
            if areas[smaller] / max(areas[larger], 1e-6) > area_ratio_thr:
                continue
            inter = np.logical_and(bins[smaller], bins[larger]).sum()
            if inter / max(areas[smaller], 1e-6) >= containment_thr:
                keep[smaller] = False
    return [int(k) for k in np.flatnonzero(keep)]

# ------------ YOLOv8-seg GT reader ------------
def read_yolo_seg_masks(label_path: Path, img_w: int, img_h: int):
    if not label_path.exists():
        return []
    text = label_path.read_text().strip()
    if not text:
        return []

    h, w = img_h, img_w
    masks = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 7:          # class id + at least 3 points
            continue
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue
        xs = coords[0::2]
        ys = coords[1::2]
        pts = []
        for x_norm, y_norm in zip(xs, ys):
            x = int(x_norm * w)
            y = int(y_norm * h)
            pts.append([x, y])
        pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 255)
        masks.append(mask)

    return masks

## Step 1 · Select FPs, check the polygon mapping, draw contact sheets
- `pred_id` is the instance id. Its line in the B3 label file follows the row order of `B3/export_stats.csv`, which is the order the lines were written.
- **Check:** for every selected FP, the notebook recomputes `best_iou_any` from the polygon and the GT, and stops if it differs from the CSV.
- **Tile:** crop with the predicted polygon in **red** and the GT polygons in **green**, next to a whole-image thumbnail (FP box in red, GT outlines in green). The label underneath shows: running ID, image name, `best_iou_any`.

In [ ]:
# ============================================================
# Cell 3 — Step 1: selection, mapping check, contact sheets, fp_tags.csv
# ============================================================
def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

rows = list(csv.DictReader(open(B3_DIR / "per_instance.csv", encoding="utf-8")))
fp_rows = [r for r in rows if r["outcome"] == "FP"]
assert len(fp_rows) == B3_METRICS["FP"] == 114, f"expected 114 FP, found {len(fp_rows)}"
selected = [r for r in fp_rows if float(r["best_iou_any"]) <= FP_IOU_MAX]
boundary = [r for r in fp_rows if FP_IOU_MAX < float(r["best_iou_any"]) < 0.5]
assert len(selected) + len(boundary) == len(fp_rows)
COUNTS = {"fp_total": len(fp_rows), "selected_le_0.1": len(selected),
          "selected_exactly_0": sum(float(r["best_iou_any"]) == 0 for r in selected),
          "selected_between_0_and_0.1": sum(float(r["best_iou_any"]) > 0 for r in selected),
          "boundary_0.1_to_0.5": len(boundary)}
print(COUNTS)

# line index of each prediction inside the B3 label file
line_of = {}
_order = {}
for r in csv.DictReader(open(B3_DIR / "export_stats.csv", encoding="utf-8")):
    _order.setdefault(r["image"], []).append(r["pred_id"])
for img, ids in _order.items():
    for k, pid in enumerate(ids):
        line_of[(img, pid)] = k

gt_label_by_image = {}
for lbl in sorted(GT_LABELS_DIR.glob("*.txt")):
    p = label_to_image_path(lbl, EVAL_IMG_DIR)
    if p is not None:
        gt_label_by_image[p.name] = (lbl, p)
assert len(gt_label_by_image) == 200


def polygon_px(line, w, h):
    """Same coordinate rule as read_yolo_seg_masks (int of normalised * size)."""
    v = [float(x) for x in line.split()[1:]]
    return np.array([[int(x * w), int(y * h)] for x, y in zip(v[0::2], v[1::2])], dtype=np.int32)


def load_fp(r):
    lbl, img_path = gt_label_by_image[r["image"]]
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    lines = [l for l in (B3_DIR / "labels" / f"{Path(r['image']).stem}.txt").read_text().splitlines() if l.strip()]
    line = lines[line_of[(r["image"], r["pred_id"])]]
    pred_poly = polygon_px(line, w, h)
    gt_lines = [l for l in lbl.read_text().splitlines() if len(l.split()) >= 7]
    gt_polys = [polygon_px(l, w, h) for l in gt_lines]
    # mapping check: rasterise exactly like the evaluation and recompute best IoU and area
    pred_mask = np.zeros((h, w), np.uint8); cv2.fillPoly(pred_mask, [pred_poly.reshape(-1, 1, 2)], 255)
    gt_masks = read_yolo_seg_masks(lbl, w, h)
    best = max([mask_iou(g, pred_mask) for g in gt_masks], default=0.0)
    assert abs(best - float(r["best_iou_any"])) < 1e-5, f"mapping check failed for {r['image']} pred {r['pred_id']}"
    assert int((pred_mask > 0).sum()) == int(r["pred_area_px"]), f"area check failed for {r['image']} pred {r['pred_id']}"
    return img, pred_poly, gt_polys


def make_tile(fp_id, r):
    img, pred_poly, gt_polys = load_fp(r)
    h, w = img.shape[:2]
    x0, y0 = pred_poly.min(axis=0); x1, y1 = pred_poly.max(axis=0)
    side = max(int(round(max(x1 - x0 + 1, y1 - y0 + 1) * (1 + CROP_PAD_FRAC))), CROP_MIN_PX)
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    ox, oy = int(round(cx - side / 2.0)), int(round(cy - side / 2.0))
    # crop with grey padding where the square goes past the image border
    canvas = np.full((side, side, 3), 114, np.uint8)
    sx0, sy0, sx1, sy1 = max(ox, 0), max(oy, 0), min(ox + side, w), min(oy + side, h)
    canvas[sy0 - oy:sy1 - oy, sx0 - ox:sx1 - ox] = img[sy0:sy1, sx0:sx1]
    s = TILE_PX / side
    crop = cv2.resize(canvas, (TILE_PX, TILE_PX), interpolation=cv2.INTER_AREA)
    to_tile = lambda P: np.round((P - [ox, oy]) * s).astype(np.int32).reshape(-1, 1, 2)
    for g in gt_polys:
        cv2.polylines(crop, [to_tile(g)], True, (0, 255, 0), 2)
    cv2.polylines(crop, [to_tile(pred_poly)], True, (0, 0, 255), 2)

    t = THUMB_PX / max(h, w)
    thumb = np.full((THUMB_PX, THUMB_PX, 3), 255, np.uint8)
    th = cv2.resize(img, (int(round(w * t)), int(round(h * t))), interpolation=cv2.INTER_AREA)
    thumb[:th.shape[0], :th.shape[1]] = th
    for g in gt_polys:
        cv2.polylines(thumb, [np.round(g * t).astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    cv2.rectangle(thumb, (int(x0 * t), int(y0 * t)), (int(x1 * t), int(y1 * t)), (0, 0, 255), 2)

    tile = np.full((TILE_PX + 40, TILE_PX + THUMB_PX + 6, 3), 255, np.uint8)
    tile[:TILE_PX, :TILE_PX] = crop
    tile[:THUMB_PX, TILE_PX + 6:] = thumb
    cv2.putText(tile, fp_id, (4, TILE_PX + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(tile, f"{r['image']}  IoU {float(r['best_iou_any']):.2f}", (4, TILE_PX + 34),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    return cv2.copyMakeBorder(tile, 0, TILE_GAP_PX, 0, TILE_GAP_PX, cv2.BORDER_CONSTANT, value=(255, 255, 255))


def build_sheets(selected, out_dir, max_sheets=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    per = GRID_COLS * GRID_ROWS
    n_sheets = (len(selected) + per - 1) // per
    if max_sheets:
        n_sheets = min(n_sheets, max_sheets)
    paths = []
    for s in range(n_sheets):
        tiles = [make_tile(f"FP{i + 1:03d}", r) for i, r in enumerate(selected) if s * per <= i < (s + 1) * per]
        blank = np.full_like(tiles[0], 255)
        tiles += [blank] * (per - len(tiles))
        grid = cv2.vconcat([cv2.hconcat(tiles[k * GRID_COLS:(k + 1) * GRID_COLS]) for k in range(GRID_ROWS)])
        p = out_dir / f"sheet_{s + 1:02d}.png"
        cv2.imwrite(str(p), grid)
        paths.append(p)
        print("Saved", p, f"(FP{s * per + 1:03d}-FP{min((s + 1) * per, len(selected)):03d})")
    return paths


# Fixed order: by image name, then by line order in the label file
selected.sort(key=lambda r: (r["image"], line_of[(r["image"], r["pred_id"])]))
SHEETS = build_sheets(selected, SHEET_DIR, MAX_SHEETS)

if TAGS_CSV.exists():
    print(f"{TAGS_CSV.name} already exists; NOT overwritten (your tags are safe).")
else:
    with open(TAGS_CSV, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=["fp_id", "image", "pred_id", "best_iou_any", "pred_area_px", "tag", "note"])
        wr.writeheader()
        for i, r in enumerate(selected):
            wr.writerow({"fp_id": f"FP{i + 1:03d}", "image": r["image"], "pred_id": r["pred_id"],
                         "best_iou_any": r["best_iou_any"], "pred_area_px": r["pred_area_px"], "tag": "", "note": ""})
    print("Saved", TAGS_CSV, f"({len(selected)} rows, tag/note blank)")

(FP_DIR / "provenance_step1.json").write_text(json.dumps({
    "notebook_commit": GIT_COMMIT, "notebook_version": NOTEBOOK_VERSION, "source_run_commit": SOURCE_COMMIT,
    "source_per_instance_sha256": sha256_file(B3_DIR / "per_instance.csv"),
    "selection_rule": f"B3 FP rows with best_iou_any <= {FP_IOU_MAX}", "counts": COUNTS,
    "order": "image name, then line order in the B3 label file",
    "tile": {"crop": f"bbox x{1 + CROP_PAD_FRAC} (square, min {CROP_MIN_PX} px) -> {TILE_PX}x{TILE_PX}",
             "pred": "red outline", "gt": "green outline", "thumbnail": f"{THUMB_PX} px, FP box in red, GT outlines in green"},
    "mapping_check": "best_iou_any and pred_area_px recomputed from each polygon; all matched",
    "created": datetime.now().isoformat(timespec="seconds"),
}, indent=2))

## Step 3 · Summary (run after tagging `fp_tags.csv`)
This step needs every row to have exactly one tag from `ROAD, PATCH, OBJ, UA, UNC`. The precision counting UA as correct is an **upper bound for sensitivity only**, not a replacement metric.

In [ ]:
# ============================================================
# Cell 4 — Step 3: count table, UA upper bound, examples
# ============================================================
def summarise_tags():
    tagged = list(csv.DictReader(open(TAGS_CSV, encoding="utf-8-sig")))
    assert len(tagged) == COUNTS["selected_le_0.1"], f"fp_tags.csv has {len(tagged)} rows, expected {COUNTS['selected_le_0.1']}"
    ids = [(r["image"], r["pred_id"]) for r in tagged]
    assert ids == [(r["image"], r["pred_id"]) for r in selected], "fp_tags.csv rows changed order or content"
    tags = [r["tag"].strip().upper() for r in tagged]
    bad = [(r["fp_id"], r["tag"]) for r, t in zip(tagged, tags) if t not in ALLOWED_TAGS]
    if bad:
        print(f"{len(bad)} rows have a missing or invalid tag (allowed: {ALLOWED_TAGS}), e.g. {bad[:5]}")
        return None

    n_fp, TP = COUNTS["fp_total"], B3_METRICS["TP"]
    table = []
    for t in ALLOWED_TAGS:
        n = tags.count(t)
        table.append({"category": t, "n": n, "pct_of_all_FP": round(100 * n / n_fp, 1),
                      "example_ids": " ".join([r["fp_id"] for r, x in zip(tagged, tags) if x == t][:2])})
    table.append({"category": "BOUNDARY (0.1 < best_iou_any < 0.5)", "n": COUNTS["boundary_0.1_to_0.5"],
                  "pct_of_all_FP": round(100 * COUNTS["boundary_0.1_to_0.5"] / n_fp, 1), "example_ids": ""})
    table.append({"category": "TOTAL", "n": sum(r["n"] for r in table), "pct_of_all_FP": 100.0, "example_ids": ""})
    assert table[-1]["n"] == n_fp

    n_ua = tags.count("UA")
    p_reported = TP / (TP + n_fp)
    p_ua_upper = (TP + n_ua) / (TP + n_fp)
    with open(FP_DIR / "fp_tag_summary.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(table[0].keys())); wr.writeheader(); wr.writerows(table)
    out = {"table": table, "precision_reported": p_reported,
           "precision_if_UA_counted_correct_UPPER_BOUND": p_ua_upper,
           "note": "Upper-bound sensitivity only; the reported precision is unchanged. GT, metrics and matching untouched.",
           "all_ids_per_tag": {t: [r["fp_id"] for r, x in zip(tagged, tags) if x == t] for t in ALLOWED_TAGS},
           "fp_tags_sha256": sha256_file(TAGS_CSV), "notebook_commit": GIT_COMMIT,
           "source_run_commit": SOURCE_COMMIT, "created": datetime.now().isoformat(timespec="seconds")}
    (FP_DIR / "fp_tag_summary.json").write_text(json.dumps(out, indent=2))

    print(f"{'category':40s} {'n':>4s} {'% FP':>6s}  examples")
    for r in table:
        print(f"{r['category']:40s} {r['n']:4d} {r['pct_of_all_FP']:6.1f}  {r['example_ids']}")
    print(f"\nReported precision: {p_reported:.3f}")
    print(f"UPPER BOUND if UA counted correct (sensitivity only): ({TP} + {n_ua}) / ({TP} + {n_fp}) = {p_ua_upper:.3f}")
    return out


if TAGS_CSV.exists():
    FP_SUMMARY = summarise_tags()